In [3]:
import torch
from stable_baselines3 import PPO
import numpy as np

# 1. Cargar el modelo
model = PPO.load("models/ppo_bala2fire_1.zip")


def to_cpp_array(name, array):
    """Convierte un numpy array en una cadena de texto formateada para C++"""
    shape = array.shape
    if len(shape) == 2:  # Matriz de pesos
        rows = ",\n    ".join(["{" + ", ".join([f"{v}f" for v in row]) + "}" for row in array])
        return f"const float {name}[{shape[0]}][{shape[1]}] = {{\n    {rows}\n}};"
    else:  # Vector de sesgo (bias)
        vals = ", ".join([f"{v}f" for v in array])
        return f"const float {name}[{shape[0]}] = {{{vals}}};"

# 2. Extraer capas de la Red de Política (Actor)
# Usamos .detach().cpu().numpy() para mover los datos a la memoria RAM normal
policy_net = model.policy.mlp_extractor.policy_net
action_net = model.policy.action_net

with open("model_data.h", "w") as f:
    f.write("// Pesos del modelo Bala2 generados automaticamente\n\n")
    
    # Capa 1: [32 x n_inputs]
    f.write(to_cpp_array("layer1_w", policy_net[0].weight.detach().cpu().numpy()) + "\n\n")
    f.write(to_cpp_array("layer1_b", policy_net[0].bias.detach().cpu().numpy()) + "\n\n")
    
    # Capa 2: [32 x 32]
    # Nota: policy_net[1] es la activacion, los pesos estan en [2]
    f.write(to_cpp_array("layer2_w", policy_net[2].weight.detach().cpu().numpy()) + "\n\n")
    f.write(to_cpp_array("layer2_b", policy_net[2].bias.detach().cpu().numpy()) + "\n\n")
    
    # Capa de Salida: [n_actions x 32]
    f.write(to_cpp_array("output_w", action_net.weight.detach().cpu().numpy()) + "\n\n")
    f.write(to_cpp_array("output_b", action_net.bias.detach().cpu().numpy()) + "\n\n")

print("¡Archivo model_data.h generado con éxito!")

¡Archivo model_data.h generado con éxito!
